# Jupyter Notebook UI to analyze tap speed screen data from tap-habituation experiments!

### Beginner Essentials:
1. Shift-Enter to run each cell. After you run, you should see an output "done step #". If not, an error has occured.
2. When inputting your own code/revising the code, make sure you close all your quotation marks `''` and brackets `()`, `[]`, `{}`.
3. Don't leave any commas hanging. If there is nothing after a comma, remove the comma.
4. Run all cells sequentially, even the ones that do not need input.


## Step-by-Step Analysis of the Jupyter Notebook

| Step | Purpose | Key Actions |
|------|---------|-------------|
| **1. Import Packages** | Load required Python libraries for data analysis | Imports `pandas`, `numpy`, widgets, plotting packages, and helpers for file discovery |
| **2. Pick Filepath** | Select the folder containing experimental data files | Uses the existing `FileChooser` widget to select the directory |
| **3. User-Defined Variables** | Set experiment parameters | Defines bins, tap timing, and tap tolerances |
| **4. Construct Filelist** | Find target `.dat` files in the selected folder | Keeps only `.dat` files whose names end with `.<numbers>.dat` |
| **5. Process Data Function** | Load and filter raw data | Retains `assign_taps`, `insert_plates`, and strain dictionary support while adding retention-window filtering |
| **6.1 Process Data** | Build strain dictionary | Creates `StrainNames` from the folder structure |
| **6.2 Process Data** | Process each strain | Passes each strain through `ProcessData()` and applies tap/plate annotations |
| **7. Grouping & Naming** | Combine processed data | Concatenates all strain data and assigns dataset, gene, allele, and screen labels |

### Key Notes:
- User input is required in Steps 2 and 3.
- The notebook targets `.dat` files with a numeric token immediately before `.dat`, for example `sample.00065.dat`.


# 1. Importing Packages Required (No input required, just run)


In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from ipyfilechooser import FileChooser
from ipywidgets import widgets
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm

pd.set_option('display.max_columns', 50)
print('done step 1')


In [ ]:
warnings.filterwarnings('ignore', category=RuntimeWarning)


## 2. Pick filepath (just run and click button from output)

Run the following cell and click the button `Select Folder` to pick a filepath.

**Important:** This notebook still uses the total filepath to infer strain names later in the pipeline.


In [ ]:
starting_directory = '/Volumes/'
chooser = FileChooser(starting_directory)
display(chooser)


In [ ]:
print(chooser.selected_path)
folder_path = chooser.selected_path


In [ ]:
screens = [
    'PD_Screen',
    'ASD_Screen',
    'G-Proteins_Screen',
    'Glia_Genes_Screen',
    'Neuron_Genes_Screen',
    'ASD_WGS_Screen',
    'PD_GWAS_Locus22_Screen',
    'PD_GWAS_Locus71_Screen',
    'Miscellaneous',
]

screen_chooser = widgets.Select(options=screens, value=screens[0], description='Screen:')
display(screen_chooser)


In [ ]:
Screen = screen_chooser.value
print(Screen)


# 3. User Defined Variables (Add input here)

Here, we add the experiment constants used throughout the notebook.


In [ ]:
bins = np.linspace(0, 1200, 1201)

number_of_taps = 30
ISI = 10
first_tap = 600

taps = np.arange(1, number_of_taps + 2).tolist()
lower = np.arange(first_tap - 0.1, first_tap - 0.1 + (number_of_taps * ISI), ISI)
upper = np.arange(first_tap + 9.9, first_tap + 9.9 + (number_of_taps * ISI), ISI)
tap_tolerances = [(float(l), float(u)) for l, u in zip(lower, upper)]
tap_tolerances.append((1199.9, 1201))

for i in taps:
    print(f'Tap {i}, tolerance: {tap_tolerances[i - 1]}')

print('done step 3')


# 4. Construct filelist from folder path (No input required, just run)


In [ ]:
os.chdir(folder_path)

file_pattern = re.compile(r'\.\d+\.dat$')
filelist = []

for root, dirs, files in os.walk(folder_path):
    for name in files:
        if file_pattern.search(name):
            filelist.append(os.path.join(root, name))

filelist = sorted(filelist)

if not filelist:
    raise FileNotFoundError('No target .dat files matching *.<numbers>.dat were found in the selected folder!')

print(f'Number of .dat files to process: {len(filelist)}')
print(f'Example file: {filelist[0]}')
print('done step 4')


# 5. Process Data Function (No input required, just run)


In [ ]:
def filter_retention_windows(df):
    """
    Retain windows that begin when Tap transitions from 0 to 1 while Bias is
    non-negative, and end just before Bias transitions from <= 0 to > 0.
    """
    working_df = df.reset_index(drop=False)
    retained_segments = []
    start_pos = None
    prev_tap = None
    prev_bias = None

    for pos, row in working_df.iterrows():
        tap_transition = prev_tap == 0 and row['Tap'] == 1
        bias_non_negative = row['Bias'] >= 0
        bias_to_positive = prev_bias is not None and prev_bias <= 0 and row['Bias'] > 0

        if start_pos is None:
            if tap_transition and bias_non_negative:
                start_pos = pos
        else:
            if bias_to_positive:
                retained_segments.append(working_df.iloc[start_pos:pos].copy())
                start_pos = None
                if tap_transition and bias_non_negative:
                    start_pos = pos

        prev_tap = row['Tap']
        prev_bias = row['Bias']

    if start_pos is not None:
        retained_segments.append(working_df.iloc[start_pos:].copy())

    if not retained_segments:
        return df.iloc[0:0].copy()

    retained_df = pd.concat(retained_segments, ignore_index=False)
    retained_df = retained_df.sort_values('index', kind='stable')
    retained_df = retained_df.drop(columns='index').reset_index(drop=True)
    return retained_df


def ProcessData(strain, experiment_counter):
    """
    Filters and processes .dat files matching the given strain.
    """
    strain_filelist = [x for x in filelist if strain in x]
    Strain_N = len(strain_filelist)

    if Strain_N == 0:
        raise AssertionError(f'{strain} is not a good identifier')

    print(f'Strain {strain}')
    print(f'Number of worms: {Strain_N}')

    df_list = []
    for file in tqdm(strain_filelist):
        if file.split('/')[-1].startswith('._'):
            continue

        try:
            # print(f'File: {file}')
            df = pd.read_csv(file, sep=' ', header=None, encoding_errors='ignore')
            df = df.rename(
                {
                    0: 'Time',
                    1: 'n',
                    2: 'Number',
                    3: 'Speed',
                    4: 'Interval Speed',
                    5: 'Bias',
                    6: 'Tap',
                    7: 'Puff',
                    8: 'x',
                    9: 'y',
                    10: 'Morphwidth',
                    11: 'Midline',
                    12: 'Area',
                    13: 'Angular Speed',
                    14: 'Aspect Ratio',
                    15: 'Kink',
                    16: 'Curve',
                    17: 'Crab',
                    18: 'Pathlength',
                },
                axis=1,
            )
            df = filter_retention_windows(df)
            if df.empty:
                experiment_counter = experiment_counter + 1
                continue

            df['worm_id'] = file.split('/')[-1].split('.')[-2]
            df['Plate_id'] = file.split('/')[-2] + '_' + file.split('/')[-1].split('_')[-1].split('.')[0]
            df['Date'] = file.split('/')[-2].split('_')[0]
            df['Screen'] = file.split('/')[-4]
            df['Experiment'] = experiment_counter
            experiment_counter = experiment_counter + 1
            df_list.append(df)
        except Exception:
            print(f'error in file {file}')

    if not df_list:
        raise ValueError(f'No readable files were found for strain {strain}.')

    DF_Total = pd.concat(df_list, ignore_index=True)
    DF_Total['plate'] = 0

    print('---------------------------------------------------------------------------------------------------------------------------------------------------------------------------')

    return {
        'N': Strain_N,
        'Confirm': DF_Total,
        'experiment_counter': experiment_counter,
    }


def assign_taps(df, tap_tolerances):
    """
    Assign tap number to each row in the DataFrame based on time tolerances.
    """
    df['taps'] = np.nan
    if df.empty:
        return

    df.loc[df.index[0], 'taps'] = 0
    for taps, tolerance in enumerate(tap_tolerances):
        tap_lower, tap_upper = tolerance
        times_in_tap_range = df['Time'].between(tap_lower, tap_upper, inclusive='both')
        df.loc[times_in_tap_range, 'taps'] = int(taps) + 1


def insert_plates(df):
    """
    Insert a plate column into a dataframe.
    """
    if df.empty:
        df['plate'] = pd.Series(dtype='int64')
        return

    df['plate'] = (df['taps'] == 1).cumsum()


print('done step 5')


# 6.1 Process Data

Create a dictionary `StrainNames` that contains all the genotype/strain names from each file path.


In [ ]:
genotype = []
for f in filelist:
    genotype.append(f.split('/')[-3])

genotypes = np.unique(genotype).tolist()

if Screen == 'Neuron_Genes_Screen':
    genotypes.insert(0, genotypes.pop(genotypes.index('N2_XJ1')))
    genotypes.insert(0, genotypes.pop(genotypes.index('N2_N2')))
else:
    genotypes.insert(0, genotypes.pop(genotypes.index('N2')))

nstrains = list(range(1, len(genotypes) + 1))
StrainNames = {nstrains[i]: genotypes[i] for i in range(len(nstrains))}

print(f'Number of genotypes/strains in the experiment: {len(genotypes)}')
for k in list(StrainNames)[:5]:
    print(f'{k}: {StrainNames[k]}')

print('done step 6.1')


# 6.2 Process Data (just run this cell)

Pass each strain through `ProcessData()` function.


In [ ]:
DataLists = [0]
experiment_counter = 1

for s in tqdm(StrainNames.values()):
    if s != '':
        result = ProcessData(s, experiment_counter)
        DataLists.append(result['Confirm'])
        experiment_counter = result['experiment_counter']

for df in DataLists[1:]:
    assign_taps(df, tap_tolerances)
    insert_plates(df)

print('done step 6.2')


# 7. Grouping Data and Naming


In [ ]:
base = pd.concat(df.assign(dataset=StrainNames.get(i + 1)) for i, df in enumerate(DataLists[1:]))

base[['Gene', 'Allele']] = base['dataset'].str.split(pat='_', n=1, expand=True)
base['Screen'] = Screen
base['Allele'] = base['Allele'].fillna('N2')

base.head()


In [ ]:
base['taps'] = base['taps'].ffill()

In [ ]:
base

In [ ]:
len(base)

In [ ]:
base.to_csv(f'{folder_path}/{Screen}_processed_data.csv', index=False)

# Done!
